In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.colors import to_rgb
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3DCollection

TROPICAL_RAINFOREST = "#1B7766"
ITALIC_RED          = "#C25966"

TITLE_GREY  = "#888888"
SLATE_STEEL = "#4A72A4"

rcParams["font.family"]       = "serif"
rcParams["font.serif"]        = ["Palatino", "Palatino Linotype",
                                  "URW Palladio L", "Book Antiqua",
                                  "STIX Two Text", "STIXGeneral",
                                  "DejaVu Serif"]
rcParams["mathtext.fontset"]  = "stix"
rcParams["axes.linewidth"]    = 0.6
rcParams["xtick.major.width"] = 0.6
rcParams["ytick.major.width"] = 0.6
rcParams["font.size"]         = 9

def simulate_erw(n_steps, p, d=3, seed=None):
    rng = np.random.default_rng(seed)

    basis = np.zeros((2 * d, d), dtype=np.int64)
    for j in range(d):
        basis[j, j]      =  1
        basis[d + j, j]  = -1

    eta_idx = np.empty(n_steps, dtype=np.int64)

    eta_idx[0] = rng.integers(0, 2 * d)

    K_arr   = rng.integers(0, np.arange(1, n_steps), endpoint=False)
    rep_arr = rng.random(n_steps - 1) < p

    offset_arr = rng.integers(1, 2 * d, size=n_steps - 1)

    for n in range(1, n_steps):
        remembered = eta_idx[K_arr[n - 1]]
        if rep_arr[n - 1]:
            eta_idx[n] = remembered
        else:
            eta_idx[n] = (remembered + offset_arr[n - 1]) % (2 * d)

    eta = basis[eta_idx]
    positions = np.zeros((n_steps + 1, d), dtype=np.int64)
    np.cumsum(eta, axis=0, out=positions[1:])
    return positions

def plot_path(ax, positions, color, title=None,
              alpha_near=0.95, alpha_far=0.30, gamma=1.0):
    x, y, z = positions[:, 0], positions[:, 1], positions[:, 2]

    pts  = positions.reshape(-1, 1, 3)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)

    mids = 0.5 * (segs[:, 0, :] + segs[:, 1, :])
    r    = np.linalg.norm(mids, axis=1)
    r_max = r.max() if r.max() > 0 else 1.0
    t = (r / r_max) ** gamma
    alphas = alpha_near + (alpha_far - alpha_near) * t

    rgb = np.array(to_rgb(color))
    colors = np.tile(rgb, (segs.shape[0], 1))
    colors = np.concatenate([colors, alphas[:, None]], axis=1)

    lc = Line3DCollection(segs, colors=colors, linewidths=0.5,
                          capstyle="round", joinstyle="round")
    ax.add_collection3d(lc)

    ax.set_xlim(x.min(), x.max())
    ax.set_ylim(y.min(), y.max())
    ax.set_zlim(z.min(), z.max())

    ax.scatter([0], [0], [0],
               color="white", s=260, zorder=11,
               edgecolors="none",
               depthshade=False)
    ax.scatter([0], [0], [0],
               color=SLATE_STEEL, s=90, zorder=12,
               edgecolors="white", linewidths=1.4,
               depthshade=False)

    ax.set_axis_off()

def main():
    n_steps = 6000
    p_low   = 0.4
    p_high  = 0.7

    pos_low  = simulate_erw(n_steps, p_low,  d=3, seed=11)
    pos_high = simulate_erw(n_steps, p_high, d=3, seed=7)

    fig = plt.figure(figsize=(8.5, 4.4))

    ax1 = fig.add_subplot(1, 2, 1, projection="3d")
    plot_path(ax1, pos_low,  color=TROPICAL_RAINFOREST,
              title=fr"$p = {p_low}$")

    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    plot_path(ax2, pos_high, color=ITALIC_RED,
              title=fr"$p = {p_high}$")

    for ax in (ax1, ax2):
        ax.view_init(elev=22, azim=-55)

    fig.subplots_adjust(left=0.0, right=1.0, top=1.0, bottom=0.0,
                        wspace=-0.30)

    fig.text(0.30, 0.84, fr"$p = {p_low}$",
             ha="center", va="bottom",
             fontsize=11, color=TITLE_GREY)
    fig.text(0.70, 0.84, fr"$p = {p_high}$",
             ha="center", va="bottom",
             fontsize=11, color=TITLE_GREY)

    out_pdf = "/home/claude/erw3d/erw3d.pdf"
    out_png = "/home/claude/erw3d/erw3d.png"

    print(f"Wrote {out_pdf}")
    print(f"Wrote {out_png}")

    end_low  = pos_low[-1]
    end_high = pos_high[-1]
    print(f"p = {p_low}:  endpoint = {end_low},  |X_n| = {np.linalg.norm(end_low):.1f}")
    print(f"p = {p_high}: endpoint = {end_high}, |X_n| = {np.linalg.norm(end_high):.1f}")

if __name__ == "__main__":
    main()